##  Here we are doing the Setup, Imports, and Drive Connection

In [ ]:
# importing and setting up
import tensorflow as tf
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dense, Dropout, RandomFlip, RandomRotation, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
import os
import numpy as np
import cv2 #for the porpuse of preprocessing


# here we ae unzipping  and connecting to the drive
print("Connecting to Google Drive...")
from google.colab import drive
drive.mount('/content/drive')

print("Unzipping the DATA.zip file from Drive...")
ZIP_PATH = "/content/drive/MyDrive/DATA.zip"
!unzip -o -q {ZIP_PATH} -d "/content/"
print("Data is unzipped and ready.")

Connecting to Google Drive...
Mounted at /content/drive
Unzipping the DATA.zip file from Drive...
Data is unzipped and ready.


## We are defining  Variables and Custom Preprocessing

In [ ]:
#  Define the variable and the path

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_BINARY_CLASSES = 1 # here is our simple  Output neuron for sigmoid
EPOCHS_STAGE_1 = 15
EPOCHS_STAGE_2 = 10

TRAIN_DIR = "/content/DATA/Training(70%)"
VALID_DIR = "/content/DATA/Validation(20%)"
TEST_DIR = "/content/DATA/Testing(10%)"

# here is our simpel custom cv2 preprocessing functions
# and  here we are inserting the working cv2 logic
def apply_preprocessing(image_array):
    """Applies Median Blur for noise removal and CLAHE for contrast."""
    # here is out images array coems comes in  float32 (0-255)
    image_uint8 = image_array.astype(np.uint8)

    # here  we are removing the noise meaning Median Blur
    image_blur = cv2.medianBlur(image_uint8, 5)

    #here we are jsut enhancing the contrast meaning CLAHE
    image_lab = cv2.cvtColor(image_blur, cv2.COLOR_RGB2LAB)
    l_channel, a_channel, b_channel = cv2.split(image_lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    cl = clahe.apply(l_channel)
    merged = cv2.merge((cl, a_channel, b_channel))
    final_image = cv2.cvtColor(merged, cv2.COLOR_LAB2RGB)

    return final_image.astype(np.float32)

@tf.function
def tf_preprocess_wrapper(image, label):
    # Wrapper for CV2 preprocessing (0-255)
    [image,] = tf.numpy_function(apply_preprocessing, [image], [tf.float32])
    image.set_shape([IMAGE_SIZE[0], IMAGE_SIZE[1], 3])
    return image, label # here we are just returning the images and origibal 4 class label

## Data Pipeline and Binary Label Mapping

In [ ]:
#  here we are customing the data pipeline for the binary labels
@tf.function
def map_to_binary_label(image, multi_class_label):
    """
    Derives the binary label (0 or 1) from the 4-class one-hot tensor.
    FIXED: Uses index [2] for 'no_tumor'.
    """
    #  here we are getting  the 'No Tumor' value from the correct index
    no_tumor_value = multi_class_label[2]

    # Binary labeling = 1 (Tumor) - No Tumor Value
    binary_label = 1.0 - no_tumor_value

    # Ensure the binary label is a float32 tensor of shape
    binary_label = tf.expand_dims(binary_label, axis=0)
    return image, binary_label

# here we are just loading  and preparing the datasets
print("Loading Training, Validation, and Test datasets")

# Load initial data meaning Generates 4-class one-hot labels
train_dataset = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, label_mode="categorical", seed=123, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE
)
validation_dataset = tf.keras.utils.image_dataset_from_directory(
    VALID_DIR, label_mode="categorical", seed=123, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE
)
test_dataset = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR, label_mode="categorical", image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, shuffle=False
)

#  here we are applling the preprocessing and binary label mapping
print("Applying CV2 preprocessing and mapping to binary labels")
AUTOTUNE = tf.data.AUTOTUNE

def preprocess_and_map(ds):
    # here is our CV2 Preprocessing
    ds = ds.unbatch().map(tf_preprocess_wrapper, num_parallel_calls=AUTOTUNE)
    # we just labeling the binary label and mapping
    ds = ds.map(map_to_binary_label, num_parallel_calls=AUTOTUNE)
    # here we are doing the final batching
    ds = ds.batch(BATCH_SIZE).prefetch(buffer_size=AUTOTUNE)
    return ds

train_dataset_binary = preprocess_and_map(train_dataset)
validation_dataset_binary = preprocess_and_map(validation_dataset)
test_dataset_binary = preprocess_and_map(test_dataset)

Loading Training, Validation, and Test datasets...
Found 2297 files belonging to 4 classes.
Found 573 files belonging to 4 classes.
Found 394 files belonging to 4 classes.
Applying CV2 preprocessing and mapping to binary labels...


## here we are building , training , and evaluating Binary Model

In [ ]:
# Build and train Binary Model meaning  ResNet50
print("\nBuilding the Binary ResNet50 model")
base_model = ResNet50(weights='imagenet', include_top=False,
                       input_shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3))

# in this stage we are training  teh head meaning frozen base
base_model.trainable = False
inputs = Input(shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3))
x = RandomFlip('horizontal')(inputs)
x = RandomRotation(0.1)(x)
x = Lambda(preprocess_input)(x) # we are just normalizing the ResNet's
x = base_model(x, training=False)
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)

# Binary Classifier Head (1 neuron, sigmoid)
outputs = Dense(NUM_BINARY_CLASSES, activation='sigmoid')(x)
model_binary = Model(inputs, outputs)

# Compile for Stage 1 (Binary Loss)
model_binary.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("\n Starting Binary Model Training (Stage 1: Head Only) ")
history = model_binary.fit(
    train_dataset_binary,
    validation_data=validation_dataset_binary,
    epochs=EPOCHS_STAGE_1,
    verbose=1
)
print(" Binary Stage 1 Training Complete ")

# the second stage i mean this oen is  Fine-Tuning
print("Unfreezing the top 30 layers of the model")
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

# recompile for fineTuning
model_binary.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy',
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall')]
)

print("Continuing training to fine-tune the unfrozen layers")
history_finetune = model_binary.fit(
    train_dataset_binary,
    validation_data=validation_dataset_binary,
    epochs=EPOCHS_STAGE_2,
    initial_epoch=history.epoch[-1],
    verbose=1
)
print(" Binary Stage 2 Fine-Tuning Complete ")

#  Final Evaluation and Saving
print("\n Evaluating the Binary Model on the Test Set ")
results = model_binary.evaluate(test_dataset_binary, verbose=1)

# Calculating  F1-Score
metrics = {
    'loss': results[0],
    'accuracy': results[1],
    'precision': results[2],
    'recall': results[3]
}
if (metrics['precision'] + metrics['recall']) > 0:
    metrics['f1_score'] = 2 * (metrics['precision'] * metrics['recall']) / (metrics['precision'] + metrics['recall'])
else:
    metrics['f1_score'] = 0.0

print("\n BINARY Fine-Tuned Model Test Results ")
print(f"Test Loss: {metrics['loss']:.4f}")
print(f"Test Accuracy: {metrics['accuracy']:.4f}")
print(f"Test Precision: {metrics['precision']:.4f}")
print(f"Test Recall: {metrics['recall']:.4f}")
print(f"Test F1-Score: {metrics['f1_score']:.4f}")

#  we just saving the final  binary model
print("\n Saving the binary model to Google Drive ")
os.makedirs("/content/drive/MyDrive/MODELS", exist_ok=True)
model_binary.save("/content/drive/MyDrive/MODELS/resnet_binary_finetuned.h5")

print("Binary Fine-Tuned ResNet model saved. Experiment Complete.")


Building the Binary ResNet50 model...
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step

--- Starting Binary Model Training (Stage 1: Head Only) ---
Epoch 1/15
     72/Unknown 25s 176ms/step - accuracy: 0.8228 - loss: 0.4682

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:160: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


72/72 ━━━━━━━━━━━━━━━━━━━━ 31s 255ms/step - accuracy: 0.8235 - loss: 0.4664 - val_accuracy: 0.9476 - val_loss: 0.1868
Epoch 2/15
72/72 ━━━━━━━━━━━━━━━━━━━━ 20s 273ms/step - accuracy: 0.9317 - loss: 0.1901 - val_accuracy: 0.9529 - val_loss: 0.1409
Epoch 3/15
72/72 ━━━━━━━━━━━━━━━━━━━━ 14s 190ms/step - accuracy: 0.9352 - loss: 0.1589 - val_accuracy: 0.9511 - val_loss: 0.1239
Epoch 4/15
72/72 ━━━━━━━━━━━━━━━━━━━━ 21s 195ms/step - accuracy: 0.9512 - loss: 0.1261 - val_accuracy: 0.9546 - val_loss: 0.1141
Epoch 5/15
72/72 ━━━━━━━━━━━━━━━━━━━━ 20s 283ms/step - accuracy: 0.9479 - loss: 0.1361 - val_accuracy: 0.9599 - val_loss: 0.1070
Epoch 6/15
72/72 ━━━━━━━━━━━━━━━━━━━━ 14s 192ms/step - accuracy: 0.9496 - loss: 0.1292 - val_accuracy: 0.9634 - val_loss: 0.1037
Epoch 7/15
72/72 ━━━━━━━━━━━━━━━━━━━━ 14s 191ms/step - accuracy: 0.9634 - loss: 0.1127 - val_accuracy: 0.9599 - val_loss: 0.1026
Epoch 8/15
72/72 ━━━━━━━━━━━━━━━━━━━━ 14s 191ms/step - accuracy: 0.9596 - loss: 0.1047 - val_accuracy: 0.961


--- BINARY Fine-Tuned Model Test Results ---
Test Loss: 0.5199
Test Accuracy: 0.7944
Test Precision: 0.9444
Test Recall: 0.7647
Test F1-Score: 0.8451

--- Saving the binary model to Google Drive ---
Binary Fine-Tuned ResNet model saved. Experiment Complete.
